# 01 - Market Microstructure Analysis

Before computing OFI, we need to understand the data.

1. Data overview - LOBSTER format explained
2. Order book snapshot visualization
3. LOB depth heatmap (all 10 levels over time)
4. Microstructure metrics: spread, depth, queue imbalance
5. Order flow analysis: add/cancel/trade breakdown
6. Intraday patterns

In [1]:
import importlib.util, os
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')
os.makedirs('../plots', exist_ok=True)

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, os.path.abspath(path))
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

ofi_mod = load_module('ofi', '../src/ofi.py')
viz_mod = load_module('viz', '../src/visualization.py')

load_lob_data        = ofi_mod.load_lob_data
microstructure_metrics = ofi_mod.microstructure_metrics
lob_snapshot         = ofi_mod.lob_snapshot

df = load_lob_data('../data/first_25000_rows.csv')
print(f'Loaded {len(df)} events | {df.ts_event.iloc[0]} → {df.ts_event.iloc[-1]}')
print(f'Symbol: {df["symbol"].iloc[0] if "symbol" in df.columns else "AAPL"}')
df.head(3)

Loaded 5000 events | 2024-10-21 11:54:29.221064336+00:00 → 2024-10-21 13:04:20.130842270+00:00
Symbol: AAPL


,ts_recv,ts_event,rtype,publisher_id,instrument_id,action,side,depth,price,size,...,ask_px_09,bid_sz_09,ask_sz_09,bid_ct_09,ask_ct_09,symbol,mid_price,spread,spread_bps,log_mid
0,2024-10-21T11:54:29.221230963Z,2024-10-21 11:54:29.221064336+00:00,10,2,38,C,B,1,233.62,2,...,234.13,55,400,2,1,AAPL,233.705,0.07,2.995229,5.45406
1,2024-10-21T11:54:29.223936626Z,2024-10-21 11:54:29.223769812+00:00,10,2,38,A,B,0,233.67,2,...,234.13,55,400,2,1,AAPL,233.705,0.07,2.995229,5.45406
2,2024-10-21T11:54:29.225196809Z,2024-10-21 11:54:29.225030400+00:00,10,2,38,A,B,0,233.67,3,...,234.13,55,400,2,1,AAPL,233.705,0.07,2.995229,5.45406


## 1. Data Overview - LOBSTER Format

In [2]:
print('=== LOBSTER DATA FORMAT ===')
print(f'Total events:  {len(df):,}')
print(f'LOB levels:    {ofi_mod.get_lob_levels(df)}')
print(f'Columns:       {len(df.columns)}')
print()
if 'action' in df.columns:
    print('Event types (action):')
    for act, cnt in df['action'].value_counts().items():
        labels = {'A': 'Add order', 'C': 'Cancel order',
                  'T': 'Trade/execution', 'D': 'Delete', 'F': 'Fill'}
        print(f'  {act} ({labels.get(act, act)}): {cnt:,} ({cnt/len(df)*100:.1f}%)')
print()
if 'side' in df.columns:
    print('Order sides:')
    for side, cnt in df['side'].value_counts().items():
        labels = {'B': 'Bid (buy)', 'A': 'Ask (sell)', 'N': 'None'}
        print(f'  {side} ({labels.get(side, side)}): {cnt:,}')

=== LOBSTER DATA FORMAT ===
Total events:  5,000
LOB levels:    10
Columns:       78

Event types (action):
  A (Add order): 2,695 (53.9%)
  C (Cancel order): 2,019 (40.4%)
  T (Trade/execution): 286 (5.7%)

Order sides:
  B (Bid (buy)): 2,497
  A (Ask (sell)): 2,383
  N (None): 120


## 2. Microstructure Metrics

In [3]:
metrics = microstructure_metrics(df)
print('=== MICROSTRUCTURE METRICS ===')
for k, v in metrics.items():
    print(f'  {k:25s}: {v}')

=== MICROSTRUCTURE METRICS ===
  n_events                 : 5000
  duration_s               : 4190.9
  arrival_rate_hz          : 0.64
  n_adds                   : 2695
  n_cancels                : 2019
  n_trades                 : 286
  mean_spread              : 0.1253
  median_spread            : 0.11
  mean_spread_bps          : 5.36
  mean_depth               : 409.5
  mean_queue_imb           : -0.0059
  mid_price_vol_ann        : 13.45
  price_range              : 0.54


## 3. Spread & Depth Analysis

In [4]:
fig = viz_mod.spread_depth_chart(df)
fig.show()

## 4. LOB Depth Heatmap

In [5]:
fig2 = viz_mod.lob_heatmap(df, n_levels=5, sample_every=5)
fig2.show()
print('The heatmap shows how bid/ask liquidity evolves over time.')
print('Bright = high depth (more liquidity), dark = low depth.')

The heatmap shows how bid/ask liquidity evolves over time.
Bright = high depth (more liquidity), dark = low depth.


## 5. LOB Snapshot at a Specific Time

In [6]:
# Show LOB at midpoint of data
snap_idx = len(df) // 2
snap = lob_snapshot(df, snap_idx, n_levels=5)
print(f'LOB Snapshot at {snap["timestamp"]}')
print(f'Mid Price: ${snap["mid_price"]:.4f}  |  Spread: ${snap["spread"]:.4f}')
print()

# Visual LOB
bids = pd.DataFrame(snap['bids'])
asks = pd.DataFrame(snap['asks'])

fig3 = go.Figure()
fig3.add_trace(go.Bar(
    y=[f'Bid L{i}' for i in range(len(bids))],
    x=bids['size'], orientation='h',
    marker_color='#2F9E44', opacity=0.8, name='Bid',
    text=[f'${p:.2f}' for p in bids['price']], textposition='auto'
))
fig3.add_trace(go.Bar(
    y=[f'Ask L{i}' for i in range(len(asks))],
    x=asks['size'], orientation='h',
    marker_color='#E03131', opacity=0.8, name='Ask',
    text=[f'${p:.2f}' for p in asks['price']], textposition='auto'
))
fig3.update_layout(
    title=f'LOB Snapshot — {snap["timestamp"]}',
    barmode='group', xaxis_title='Size',
    height=400, template='plotly_white'
)
fig3.show()

LOB Snapshot at 2024-10-21 12:29:09.096640225+00:00
Mid Price: $233.8900  |  Spread: $0.0600



## 6. Order Flow Timeline

In [8]:
if 'action' in df.columns:
    fig4 = make_subplots(rows=2, cols=2,
                          subplot_titles=['Events Over Time', 'Event Type Distribution',
                                          'Trade Size Distribution', 'Inter-event Time (ms)'])
    C = ['#4C6EF5','#F76707','#2F9E44','#E03131','#7950F2']

    # Events over time
    for act, color in zip(['A','C','T'], C):
        sub = df[df['action'] == act]
        if len(sub) > 0:
            fig4.add_trace(go.Scatter(
                x=sub['ts_event'], y=[act]*len(sub),
                mode='markers', marker=dict(size=3, color=color, opacity=0.5),
                name=act
            ), row=1, col=1)

    # Pie chart
    counts = df['action'].value_counts()
    fig4 = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Events Over Time', 'Event Type Distribution',
        'Trade Size Distribution', 'Inter-event Time (ms)'
    ],
    specs=[
        [{"type": "xy"}, {"type": "domain"}],  # 👈 pie here
        [{"type": "xy"}, {"type": "xy"}]
    ]
)

    # Trade size distribution
    trades = df[df['action'] == 'T']
    if 'size' in trades.columns and len(trades) > 0:
        fig4.add_trace(go.Histogram(
            x=trades['size'], nbinsx=30,
            marker_color=C[3], opacity=0.8, name='Trade Size'
        ), row=2, col=1)

    # Inter-event time
    iet = df['ts_event'].diff().dt.total_seconds() * 1000  # ms
    iet = iet[iet > 0].clip(0, iet.quantile(0.99))
    fig4.add_trace(go.Histogram(
        x=iet, nbinsx=50,
        marker_color=C[0], opacity=0.8, name='IET (ms)'
    ), row=2, col=2)

    fig4.update_layout(title='Order Flow Analysis', height=550,
                        template='plotly_white', showlegend=False)
    fig4.show()
    mean_iet = iet.mean()
    print(f'Mean inter-event time: {mean_iet:.2f} ms  ({1000/mean_iet:.0f} events/second)')

Mean inter-event time: 823.41 ms  (1 events/second)
